# 54. SWEAP / PSP — Implementation / 구현

**Paper**: Kasper, J. C., Abiad, R., Austin, G., et al. (2016). "Solar Wind Electrons Alphas and Protons (SWEAP) Investigation: Design of the Solar Wind and Coronal Plasma Instrument Suite for Solar Probe Plus", *Space Science Reviews* **204**, 131-186. DOI: 10.1007/s11214-015-0206-3.

This notebook reproduces, in toy form, the four signature physics/engineering analyses behind the SWEAP design at the final perihelion of 9.86 R$_\odot$:

이 노트북은 SWEAP 설계를 뒷받침하는 네 가지 대표 분석을 9.86 R$_\odot$ 근일점 조건에서 toy 모델로 재현합니다.

1. **Part 1**: Solar Probe Cup (SPC) Faraday-cup I-V curve in the 9.86 R$_\odot$ environment (high $T_p$, super-Alfvénic flow with spacecraft aberration) / SPC Faraday Cup의 I-V 곡선과 흐름 각 응답.
2. **Part 2**: SPAN top-hat ESA velocity-space response — energy/charge and elevation-deflection acceptance / SPAN top-hat ESA의 속도공간 응답.
3. **Part 3**: Kinetic non-Maxwellian VDF moment integration — recover $n$, $\mathbf{V}$, $T$ from a synthetic core+beam $f(\mathbf{v})$ / 비-맥스웰 VDF에서 모멘트 (밀도, 속도, 온도) 적분 복원.
4. **Part 4**: Thermal protection — radiative balance of SPC components under the 1400 °C heat-shield rim / 1400 °C 차폐막 가장자리에서 SPC 구성 요소의 복사 평형.

All data are synthetic; no real PSP/SWEAP telemetry is downloaded.

모든 데이터는 합성이며 실제 PSP/SWEAP 텔레메트리는 사용되지 않습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants
from scipy.integrate import quad, simpson

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

rng = np.random.default_rng(seed=54)

# Physical constants (SI).
K_B = constants.k                     # Boltzmann constant [J/K]
M_P = constants.proton_mass           # proton mass [kg]
Q_E = constants.elementary_charge     # electron charge [C]
EV_TO_J = Q_E                         # 1 eV in joules
SIGMA_SB = constants.Stefan_Boltzmann  # Stefan-Boltzmann constant

print(f'k_B = {K_B:.3e} J/K')
print(f'm_p = {M_P:.3e} kg')
print(f'sigma_SB = {SIGMA_SB:.3e} W m^-2 K^-4')

## Part 1: SPC Faraday-Cup I-V Curve at 9.86 R$_\odot$ / 9.86 R$_\odot$에서 SPC I-V 곡선

The Solar Probe Cup (SPC) is a sun-staring Faraday cup mounted at the rim of the heat shield (TPS). Its modulator grid oscillates between $V_\text{low}$ and $V_\text{high}$ at $f_\text{mod} = 1280$ Hz. Only ions with parallel kinetic energy per charge $E_\parallel/q \in [V_\text{low}, V_\text{high}]$ reach the four-quadrant collector plates, producing a measurable plate current

$$ I_\text{plate}(V) \;=\; q\,A_\text{eff}\, \int_{v_\text{low}}^{\infty} v_\parallel\, F(v_\parallel)\, dv_\parallel, \qquad v_\text{low} = \sqrt{2qV/m}, $$

where $F(v_\parallel)$ is the line-of-sight reduced distribution function. The differential I-V curve $-dI/dV$ is proportional to the RDF — exactly what synchronous detection at $f_\text{mod}$ extracts.

SPC는 차폐막 가장자리에 장착된 태양 직시 Faraday Cup으로, modulator 격자가 $f_\text{mod} = 1280$ Hz로 진동하며 LOS-parallel 에너지/전하가 $[V_\text{low}, V_\text{high}]$ 범위에 있는 이온만 collector에 도달합니다. I-V 곡선의 미분이 RDF에 비례합니다.

**Conditions at 9.86 R$_\odot$ (final perihelion)** (notes §4.F):

- Proton density $n_p = 1800$ cm$^{-3}$
- Bulk speed (radial in solar frame) $V_p = 285$ km s$^{-1}$
- Proton temperature $T_p = 41$ eV (highly anisotropic plasma; we use isotropic Maxwellian here)
- Spacecraft tangential velocity $V_\text{SC} = 200$ km s$^{-1}$ → aberration angle $\theta_\text{aberr} = \arctan(V_\text{SC}/V_p)$.

We synthesise the SPC I-V curve and verify the differential recovers the RDF.

9.86 R$_\odot$ 조건 ($n_p = 1800$ cm$^{-3}$, $V_p = 285$ km/s, $T_p = 41$ eV, 우주선 횡속도 200 km/s)에서 I-V 곡선을 합성하고, 미분이 RDF를 복원함을 확인합니다.

In [ ]:
def maxwellian_rdf(v_par_kms: np.ndarray,
                    n_cm3: float,
                    v_bulk_kms: float,
                    T_eV: float) -> np.ndarray:
    """Compute the 1-D reduced distribution function for a Maxwellian.

    Args:
        v_par_kms: Line-of-sight velocity sample points [km/s].
        n_cm3: Proton number density [cm^-3].
        v_bulk_kms: Bulk LOS speed [km/s].
        T_eV: Temperature [eV].

    Returns:
        F(v_par) in s/m^4 (SI), suitable to integrate against v dv to give
        a flux in m^-2 s^-1.
    """
    n_si = n_cm3 * 1.0e6              # m^-3
    v_par_si = v_par_kms * 1.0e3       # m/s
    v_bulk_si = v_bulk_kms * 1.0e3
    v_th_si = np.sqrt(2.0 * T_eV * EV_TO_J / M_P)  # thermal speed
    norm = n_si / (np.sqrt(np.pi) * v_th_si)
    arg = -((v_par_si - v_bulk_si) / v_th_si) ** 2
    return norm * np.exp(arg)


def spc_iv_curve(V_grid: np.ndarray,
                  n_cm3: float,
                  v_los_kms: float,
                  T_eV: float,
                  area_cm2: float = 5.0) -> np.ndarray:
    """Synthesise the SPC plate current vs modulator threshold voltage.

    For a high-pass modulator window with cut-off voltage V, the plate
    current is the flux integral of all ions with v_par > sqrt(2qV/m).

    Args:
        V_grid: Modulator threshold voltages [V].
        n_cm3: Proton number density [cm^-3].
        v_los_kms: Bulk LOS speed in spacecraft frame [km/s].
        T_eV: Proton temperature [eV].
        area_cm2: Effective collector area [cm^2].

    Returns:
        Plate current I(V) in amperes, of shape (len(V_grid),).
    """
    area_si = area_cm2 * 1.0e-4
    v_par_grid = np.linspace(50.0, 2000.0, 4001)  # km/s
    F = maxwellian_rdf(v_par_grid, n_cm3, v_los_kms, T_eV)
    v_par_si = v_par_grid * 1.0e3
    integrand = v_par_si * F
    currents = np.zeros_like(V_grid)
    for i, V in enumerate(V_grid):
        v_cut = np.sqrt(2.0 * Q_E * max(V, 1.0) / M_P)  # m/s
        mask = v_par_si > v_cut
        if mask.sum() < 2:
            currents[i] = 0.0
        else:
            currents[i] = Q_E * area_si * simpson(integrand[mask], x=v_par_si[mask])
    return currents


# 9.86 R_sun perihelion conditions, transformed to spacecraft frame.
n_p = 1800.0          # cm^-3
v_sw = 285.0          # km/s radial in solar frame
v_sc = 200.0          # km/s tangential
T_p = 41.0            # eV
v_los_sc = np.sqrt(v_sw ** 2 + v_sc ** 2)         # |V_obs| in SC frame
theta_aberr = np.degrees(np.arctan2(v_sc, v_sw))  # deg from radial
E_los_eV = 0.5 * M_P * (v_los_sc * 1e3) ** 2 / EV_TO_J

print(f'LOS speed in SC frame: {v_los_sc:.1f} km/s')
print(f'Aberration angle     : {theta_aberr:.1f} deg from radial')
print(f'LOS kinetic energy   : {E_los_eV:.1f} eV')

# Sweep the modulator threshold voltage from 50 V to 2000 V.
V_grid = np.linspace(50.0, 2000.0, 200)
I_V = spc_iv_curve(V_grid, n_p, v_los_sc, T_p, area_cm2=5.0)

# Differential = -dI/dV is proportional to the RDF; AC sync detection
# at f_mod measures this directly.
dI_dV = -np.gradient(I_V, V_grid)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.plot(V_grid, I_V * 1e9, 'C0-', lw=2.0)
ax.axvline(E_los_eV, ls='--', color='C3', alpha=0.7,
           label=f'E_LOS ≈ {E_los_eV:.0f} eV')
ax.set_xlabel('Modulator threshold voltage V [V]')
ax.set_ylabel('Plate current I [nA]')
ax.set_title('SPC integral I-V curve at 9.86 R$_\\odot$\nSPC 적분 I-V 곡선')
ax.set_yscale('log')
ax.set_ylim(1e-3, None)
ax.legend()
ax.grid(alpha=0.3, which='both')

ax = axes[1]
ax.plot(V_grid, dI_dV * 1e9, 'C2-', lw=2.0,
        label='-dI/dV (toy RDF proxy)')
ax.axvline(E_los_eV, ls='--', color='C3', alpha=0.7,
           label=f'E_LOS ≈ {E_los_eV:.0f} eV')
ax.set_xlabel('Modulator threshold voltage V [V]')
ax.set_ylabel('-dI/dV [nA / V]')
ax.set_title('Differential I-V → RDF\n미분 I-V → RDF')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Quick SNR check vs SPC noise floor.
I_peak = I_V.max()
noise_floor = 5e-13  # A (notes §3.2)
saturation = 1e-7    # A
print(f'Peak plate current : {I_peak:.2e} A')
print(f'SPC noise floor    : {noise_floor:.0e} A')
print(f'SPC saturation     : {saturation:.0e} A')
print(f'SNR (peak/noise)   : {I_peak / noise_floor:.1e}')
print(f'Headroom to sat.   : {saturation / I_peak:.1f}×')

### 1.b Four-quadrant flow-angle determination / 4분면 흐름 각 결정

SPC's four wedge-shaped collectors (numbered 1-4 around the cup axis) report independent currents $I_1, I_2, I_3, I_4$. The bulk flow direction in the cup frame is recovered from the current ratios (notes §3.2):

$$ \tan\theta_y \;\approx\; \frac{(I_1 + I_2) - (I_3 + I_4)}{\sum_k I_k},\qquad
\tan\theta_z \;\approx\; \frac{(I_1 + I_4) - (I_2 + I_3)}{\sum_k I_k}. $$

Each plate sees a current modulated by its own $\cos^2$ angular response (notes Fig. 19). We synthesise the four currents at the 9.86 R$_\odot$ aberration angle and verify the inversion.

SPC의 네 쐐기형 collector 전류 비율로 흐름 각을 복원합니다. 각 plate는 $\cos^2$ 각 응답을 가지므로, 9.86 R$_\odot$ aberration 조건에서 4개 전류를 합성한 뒤 위 식으로 흐름 각을 역산해 입력값과 비교합니다.

In [ ]:
def four_plate_currents(tilt_x_deg: float,
                         tilt_y_deg: float,
                         I_total: float) -> np.ndarray:
    """Synthesise the four SPC plate currents for a given flow tilt.

    The cup axis is along +z (sun-pointing). The flow vector arrives from
    +z with tilts toward +x (tilt_x_deg) and +y (tilt_y_deg).

    Args:
        tilt_x_deg: Flow tilt toward +x axis [deg].
        tilt_y_deg: Flow tilt toward +y axis [deg].
        I_total: Total integrated current arriving on all four plates [A].

    Returns:
        Length-4 array of plate currents [A] for k = 1..4.
    """
    tx = np.deg2rad(tilt_x_deg)
    ty = np.deg2rad(tilt_y_deg)
    flow = np.array([np.tan(tx), np.tan(ty), 1.0])
    flow /= np.linalg.norm(flow)
    # Each plate's optical axis is tilted from +z by 14 deg in one of four
    # cardinal directions: plate 1 = +y, 2 = -x, 3 = -y, 4 = +x.
    polar = np.deg2rad(14.0)
    plate_axes = np.array([
        [0.0, np.sin(polar), np.cos(polar)],   # plate 1 (+y tilt)
        [-np.sin(polar), 0.0, np.cos(polar)],  # plate 2 (-x tilt)
        [0.0, -np.sin(polar), np.cos(polar)],  # plate 3 (-y tilt)
        [np.sin(polar), 0.0, np.cos(polar)],   # plate 4 (+x tilt)
    ])
    cos_alpha = plate_axes @ flow
    weights = np.maximum(cos_alpha, 0.0) ** 2
    weights /= weights.sum()
    return I_total * weights


def recover_flow_angle(currents: np.ndarray) -> tuple:
    """Recover (tilt_x, tilt_y) from four plate currents.

    The current ratios depend on the flow tilt through a sin(2*polar)
    coupling; the calibration factor below converts the differential to
    a tilt angle in degrees.

    Args:
        currents: Length-4 array of plate currents [A].

    Returns:
        (tilt_x_deg, tilt_y_deg) recovered flow tilts [deg].
    """
    s = currents.sum()
    polar = np.deg2rad(14.0)
    cal = np.sin(2.0 * polar)
    # Plate 4 (+x) − plate 2 (-x) gives the tilt-x signal.
    # Plate 1 (+y) − plate 3 (-y) gives the tilt-y signal.
    ratio_x = (currents[3] - currents[1]) / s / cal
    ratio_y = (currents[0] - currents[2]) / s / cal
    return np.degrees(np.arctan(ratio_x)), np.degrees(np.arctan(ratio_y))


# Synthesise at the 9.86 R_sun aberration: flow is tilted toward +x in the
# spacecraft frame by theta_aberr deg (orbital direction).
tilt_x_true = theta_aberr
tilt_y_true = 0.0
I_total_in = I_V.max()
I_plates = four_plate_currents(tilt_x_true, tilt_y_true, I_total_in)

# Add 5e-13 A noise on each channel (notes §3.2 system noise).
I_meas = I_plates + 5e-13 * rng.standard_normal(4)
tx_rec, ty_rec = recover_flow_angle(I_meas)

print('Plate currents (A):')
for k, ic in enumerate(I_meas, start=1):
    print(f'  I_{k} = {ic:.3e}')
print(f'Input  flow tilts  : tilt_x = {tilt_x_true:6.2f} deg, '
      f'tilt_y = {tilt_y_true:6.2f} deg')
print(f'Recovered (toy)    : tilt_x = {tx_rec:6.2f} deg, '
      f'tilt_y = {ty_rec:6.2f} deg')
print('(The toy linear inversion is only valid for small tilts; the real\n'
      'SPC pipeline uses Kasper et al. 2006 χ² fitting against a forward\n'
      'model that captures the full cos² and modulator-window response.)')

# Sweep tilt_x from -25 to +25 deg (within SPC FOV) and confirm linearity.
tilt_scan = np.linspace(-25.0, 25.0, 41)
tx_recovered = np.zeros_like(tilt_scan)
for i, tt in enumerate(tilt_scan):
    Ic = four_plate_currents(tt, 0.0, I_total_in)
    Ic = Ic + 5e-13 * rng.standard_normal(4)
    tx_recovered[i], _ = recover_flow_angle(Ic)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(tilt_scan, tx_recovered, 'o-', label='Toy linear recovery')
ax.plot(tilt_scan, tilt_scan, 'k--', alpha=0.7, label='1:1')
ax.axvspan(-28, 28, color='C2', alpha=0.10, label='SPC ±28° FOV')
ax.set_xlabel('Input tilt_x [deg]')
ax.set_ylabel('Recovered tilt_x [deg]')
ax.set_title('Four-plate flow-angle reconstruction\n4분면 흐름 각 복원')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 2: SPAN Top-Hat ESA Velocity-Space Response / SPAN top-hat ESA 속도공간 응답

SPAN inherits the **top-hat hemispherical electrostatic analyzer** (Carlson et al. 1983, paper 55 in this list). Two concentric hemispheres at radii $R_1 < R_2$ held at potential difference $V$ pass particles with energy/charge

$$ \frac{E}{q} \;\approx\; \frac{V}{\Delta R / R}, \qquad \Delta R / R = 0.03 \;\;\text{(SPAN)}, $$

yielding $\sim$7% energy resolution. A pair of upper deflectors steers the cylindrically symmetric 360° planar FOV by $\pm 60°$ in elevation, producing a 240° × 120° FOV per sensor. We model:

1. The **transmission curve** $T(E/q\,;\,V)$ as a Gaussian centred on $E_0/q = V/(\Delta R/R)$ with $\Delta E / E = 0.07$.
2. The **deflection-elevation map** $T(\theta\,;\,V_\text{defl})$ — a Gaussian in elevation centred on $\theta_0 \propto V_\text{defl}$ with FWHM 7°.
3. The combined 2-D acceptance $T(E/q, \theta)$ on a velocity-space grid.

SPAN은 동심 반구 사이를 통과하는 입자만 검출합니다. 에너지 분해능 $\Delta R/R = 0.03$에서 $\sim$7%, deflector 쌍이 elevation ±60°를 스캔합니다. 변조 창과 deflection 응답을 2-D 속도공간에서 시각화합니다.

In [ ]:
def esa_energy_response(E_over_q_eV: np.ndarray,
                         V_inner: float,
                         dR_over_R: float = 0.03) -> np.ndarray:
    """Compute the top-hat ESA E/q transmission for a given inner-hemisphere V.

    Args:
        E_over_q_eV: Energy-per-charge sample points [eV].
        V_inner: Inner-hemisphere voltage [V] (so E0/q = V_inner / (dR/R)).
        dR_over_R: Hemisphere gap fraction (= 0.03 for SPAN).

    Returns:
        Transmission in [0, 1].
    """
    E0 = V_inner / dR_over_R       # eV centre
    sigma = 0.07 * E0 / 2.355      # FWHM ≈ 7% of E0
    return np.exp(-0.5 * ((E_over_q_eV - E0) / sigma) ** 2)


def esa_deflector_response(theta_deg: np.ndarray,
                            V_defl: float,
                            V_inner: float,
                            theta_max_deg: float = 60.0,
                            V_defl_max_frac: float = 1.0) -> np.ndarray:
    """Compute the deflector elevation response.

    Args:
        theta_deg: Sample elevation angles [deg].
        V_defl: Applied deflector voltage [V].
        V_inner: Inner-hemisphere voltage [V] (sets max deflection energy).
        theta_max_deg: Maximum elevation deflection [deg].
        V_defl_max_frac: V_defl/V_inner ratio that gives full theta_max.

    Returns:
        Transmission in [0, 1].
    """
    # Linear deflector calibration.
    theta_centre = theta_max_deg * (V_defl / V_inner) / V_defl_max_frac
    sigma = 7.0 / 2.355  # FWHM 7 deg per Fig. 25
    return np.exp(-0.5 * ((theta_deg - theta_centre) / sigma) ** 2)


# 1-D energy response for several modulator voltages.
E_eq_grid = np.linspace(50.0, 8000.0, 800)
V_inner_steps = [3.0, 6.0, 12.0, 24.0, 48.0, 96.0, 192.0]  # → 100 eV ... 6.4 keV

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
for V in V_inner_steps:
    T_curve = esa_energy_response(E_eq_grid, V)
    ax.plot(E_eq_grid, T_curve, lw=1.6, label=f'V_inner = {V:.1f} V')
ax.set_xlabel('E/q [eV]')
ax.set_ylabel('Transmission T(E/q)')
ax.set_title('SPAN top-hat energy windows (dR/R = 0.03 → ~7%)\nSPAN top-hat 에너지 창')
ax.set_xscale('log')
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3, which='both')

# 2-D combined response on (E/q, theta) at V_inner = 24 V (E0 = 800 eV).
V_inner = 24.0
E_grid = np.linspace(400.0, 1200.0, 200)
theta_grid = np.linspace(-60.0, 60.0, 200)
EE, TH = np.meshgrid(E_grid, theta_grid, indexing='xy')
T_E = esa_energy_response(EE, V_inner)

# Build a deflection sweep to fully cover ±60 deg at V_inner = 24 V.
V_defl_steps = np.linspace(-24.0, 24.0, 13)  # ratio in [-1, 1]
T_2d = np.zeros_like(TH)
for V_d in V_defl_steps:
    T_th = esa_deflector_response(TH, V_d, V_inner)
    T_2d = np.maximum(T_2d, T_E * T_th)

ax = axes[1]
im = ax.pcolormesh(E_grid, theta_grid, T_2d, shading='auto', cmap='viridis')
ax.set_xlabel('E/q [eV]')
ax.set_ylabel('Elevation θ [deg]')
ax.set_title('SPAN combined (E/q, θ) response at V_inner = 24 V\nSPAN 결합 응답')
fig.colorbar(im, ax=ax, label='Transmission')
plt.tight_layout()
plt.show()

# Verify the 7% energy resolution claim.
T_at = esa_energy_response(E_eq_grid, 24.0)
fwhm_mask = T_at > 0.5
fwhm_eV = E_eq_grid[fwhm_mask][-1] - E_eq_grid[fwhm_mask][0]
E0 = 24.0 / 0.03
print(f'V_inner = 24 V → E0 = {E0:.0f} eV')
print(f'FWHM             = {fwhm_eV:.1f} eV')
print(f'Δ E / E (FWHM)   = {fwhm_eV / E0 * 100:.1f}%   (target ≈ 7%)')

## Part 3: Kinetic Non-Maxwellian VDF Moment Integration / 비-맥스웰 VDF 모멘트 적분

Solar wind ion VDFs near the Sun are **highly non-Maxwellian**: a dense isotropic core, a temperature-anisotropic main population, and a field-aligned beam at $V_b \sim 0.5\,V_A$ (notes §1, Table 1). SWEAP's ground L3 pipeline (Kasper et al. 2006 procedure) recovers the standard moments

$$ n = \int f\, d^3v, \qquad \mathbf{V} = \frac{1}{n}\int \mathbf{v} f\, d^3v, \qquad T_{ij} = \frac{m}{n k_B} \int (v_i - V_i)(v_j - V_j) f\, d^3v. $$

We construct a synthetic two-component VDF (anisotropic core + drifting Maxwellian beam aligned with **B**), integrate to recover the moments numerically, and compare with the analytical input.

We then **mask** the VDF over the heat-shield FOV blockage (an anti-sun cone of half-angle 90°-25° = 65° around $-\hat{x}$) and check how much each moment is biased — the same calculation that motivated the SPC + SPAN-A combined coverage.

비-맥스웰 VDF (이방성 core + field-aligned beam)를 합성하고 모멘트 (밀도, 속도, 온도 텐서)를 적분 복원합니다. 그 다음 차폐막 FOV 블로킹 마스크를 적용해 단일 센서 누락이 모멘트에 미치는 편향을 정량화합니다 — SPC + SPAN-A 결합이 필요한 이유.

In [ ]:
def biMaxwellian_vdf(vx: np.ndarray,
                      vy: np.ndarray,
                      vz: np.ndarray,
                      n_cm3: float,
                      V_bulk_kms: tuple,
                      T_par_eV: float,
                      T_perp_eV: float,
                      b_hat: tuple = (1.0, 0.0, 0.0)) -> np.ndarray:
    """Compute a bi-Maxwellian VDF on a 3-D Cartesian velocity grid.

    Args:
        vx, vy, vz: Cartesian velocity components on a meshgrid [km/s].
        n_cm3: Density [cm^-3].
        V_bulk_kms: Bulk velocity 3-tuple [km/s].
        T_par_eV: Parallel temperature [eV].
        T_perp_eV: Perpendicular temperature [eV].
        b_hat: Magnetic-field unit vector (3-tuple).

    Returns:
        VDF f(v) in s^3 m^-6 (SI).
    """
    n_si = n_cm3 * 1.0e6
    Vb = np.array(V_bulk_kms) * 1.0e3
    bh = np.array(b_hat) / np.linalg.norm(b_hat)
    vth_par = np.sqrt(2.0 * T_par_eV * EV_TO_J / M_P)
    vth_per = np.sqrt(2.0 * T_perp_eV * EV_TO_J / M_P)
    norm = n_si / (np.pi ** 1.5 * vth_par * vth_per ** 2)
    # Velocity in SI relative to bulk.
    dvx = vx * 1.0e3 - Vb[0]
    dvy = vy * 1.0e3 - Vb[1]
    dvz = vz * 1.0e3 - Vb[2]
    v_par = dvx * bh[0] + dvy * bh[1] + dvz * bh[2]
    v_per_sq = (dvx ** 2 + dvy ** 2 + dvz ** 2) - v_par ** 2
    v_per_sq = np.maximum(v_per_sq, 0.0)
    return norm * np.exp(-(v_par / vth_par) ** 2 - v_per_sq / vth_per ** 2)


def vdf_moments(f_si: np.ndarray,
                  vx_kms: np.ndarray,
                  vy_kms: np.ndarray,
                  vz_kms: np.ndarray) -> dict:
    """Numerically integrate the moments of a 3-D VDF.

    Args:
        f_si: VDF on a (Nx, Ny, Nz) Cartesian grid [s^3/m^6].
        vx_kms, vy_kms, vz_kms: 1-D velocity axes [km/s].

    Returns:
        Dict with keys 'n_cm3', 'V_kms' (3,), 'T_eV' (scalar trace/3),
        and 'T_tensor_eV' (3, 3).
    """
    dx = (vx_kms[1] - vx_kms[0]) * 1.0e3
    dy = (vy_kms[1] - vy_kms[0]) * 1.0e3
    dz = (vz_kms[1] - vz_kms[0]) * 1.0e3
    dV = dx * dy * dz
    VX, VY, VZ = np.meshgrid(vx_kms * 1e3, vy_kms * 1e3, vz_kms * 1e3,
                              indexing='ij')
    n_si = (f_si * dV).sum()
    Vx = (VX * f_si * dV).sum() / n_si
    Vy = (VY * f_si * dV).sum() / n_si
    Vz = (VZ * f_si * dV).sum() / n_si
    Vvec = np.array([Vx, Vy, Vz])
    components = {0: VX, 1: VY, 2: VZ}
    T_tensor = np.zeros((3, 3))
    for i in range(3):
        for j in range(3):
            T_tensor[i, j] = (M_P / (n_si * K_B)) * (
                (components[i] - Vvec[i])
                * (components[j] - Vvec[j])
                * f_si * dV).sum()
    T_eV = (np.trace(T_tensor) / 3.0) * (K_B / EV_TO_J)
    return {
        'n_cm3': n_si / 1.0e6,
        'V_kms': Vvec / 1.0e3,
        'T_eV': T_eV,
        'T_tensor_eV': T_tensor * (K_B / EV_TO_J),
    }


# Build a velocity grid in the spacecraft frame around the bulk flow.
v_axis = np.linspace(-600.0, 200.0, 81)  # km/s (radial axis x)
vy_axis = np.linspace(-300.0, 500.0, 81)
vz_axis = np.linspace(-300.0, 300.0, 81)
VX, VY, VZ = np.meshgrid(v_axis, vy_axis, vz_axis, indexing='ij')

# Core: anisotropic, T_perp/T_par = 1.5, V_bulk = (-285, +200, 0) km/s
f_core = biMaxwellian_vdf(VX, VY, VZ,
                            n_cm3=1500.0,
                            V_bulk_kms=(-285.0, 200.0, 0.0),
                            T_par_eV=30.0,
                            T_perp_eV=50.0,
                            b_hat=(-1.0, 0.0, 0.0))
# Beam: 300 cm^-3, drifting at +0.5 V_A relative to core along B.
V_alfven = 200.0  # km/s (toy)
f_beam = biMaxwellian_vdf(VX, VY, VZ,
                            n_cm3=300.0,
                            V_bulk_kms=(-285.0 - 0.5 * V_alfven, 200.0, 0.0),
                            T_par_eV=80.0,
                            T_perp_eV=80.0,
                            b_hat=(-1.0, 0.0, 0.0))

f_total = f_core + f_beam

moments = vdf_moments(f_total, v_axis, vy_axis, vz_axis)
print('Recovered moments (full VDF, no FOV mask):')
print(f'  n      = {moments["n_cm3"]:.1f} cm^-3   '
      f'(input n_core + n_beam = {1500 + 300})')
print(f'  V      = ({moments["V_kms"][0]:7.1f}, '
      f'{moments["V_kms"][1]:7.1f}, '
      f'{moments["V_kms"][2]:7.1f}) km/s')
print(f'  T (avg) = {moments["T_eV"]:.2f} eV   '
      f'(core 36.7 eV + beam contributes broadening)')

# 2-D slice through v_z = 0 to visualise the core+beam structure.
iz0 = np.argmin(np.abs(vz_axis))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
im = ax.pcolormesh(v_axis, vy_axis, np.log10(f_total[:, :, iz0].T + 1e-25),
                    shading='auto', cmap='inferno')
ax.set_xlabel('v_x [km/s]  (sun-pointing)')
ax.set_ylabel('v_y [km/s]  (orbital)')
ax.set_title('Synthetic core+beam VDF, v_z = 0\n합성 core+beam VDF')
fig.colorbar(im, ax=ax, label='log₁₀ f [SI]')
ax.scatter([-285.0], [200.0], marker='+', color='cyan', s=120, label='core bulk')
ax.scatter([-285.0 - 0.5 * V_alfven], [200.0],
           marker='x', color='lime', s=120, label='beam bulk')
ax.legend(loc='upper right', fontsize=9)

# Apply a heat-shield FOV mask: only keep velocities arriving from
# directions within ±28° of the +x (sunward) axis (SPC FOV).
v_mag = np.sqrt(VX ** 2 + VY ** 2 + VZ ** 2)
cos_alpha = np.where(v_mag > 1e-3, -VX / v_mag, 0.0)  # +x is sunward
spc_fov_mask = cos_alpha > np.cos(np.deg2rad(28.0))
f_spc_only = np.where(spc_fov_mask, f_total, 0.0)

moments_spc = vdf_moments(f_spc_only, v_axis, vy_axis, vz_axis)
print('\nRecovered moments (SPC ±28° FOV only — single-sensor case):')
print(f'  n      = {moments_spc["n_cm3"]:.1f} cm^-3   '
      f'(true 1800)')
print(f'  V_x    = {moments_spc["V_kms"][0]:.1f} km/s   '
      f'(true −285)')
print(f'  V_y    = {moments_spc["V_kms"][1]:.1f} km/s   '
      f'(true +200)')
print(f'  T      = {moments_spc["T_eV"]:.2f} eV')

# The bias is biggest in V_y (orbital direction) — exactly what motivates
# combining SPC with the ram-side SPAN-A.
ax = axes[1]
ax.pcolormesh(v_axis, vy_axis,
              np.log10(f_spc_only[:, :, iz0].T + 1e-25),
              shading='auto', cmap='inferno')
ax.set_xlabel('v_x [km/s]')
ax.set_ylabel('v_y [km/s]')
ax.set_title('VDF after SPC ±28° FOV mask\nSPC FOV 마스크 적용 후 VDF')
ax.set_xlim(v_axis[0], v_axis[-1])
ax.set_ylim(vy_axis[0], vy_axis[-1])
plt.tight_layout()
plt.show()

n_loss = 100.0 * (1.0 - moments_spc['n_cm3'] / moments['n_cm3'])
print(f'\nFractional density loss with SPC alone: {n_loss:.1f}%')
print('At this aberration angle (~35°) much of the core sits at the edge '
      'of the ±28° SPC FOV, so SPC alone biases all moments — exactly the\n'
      'condition that requires the SPAN-A ram sensor to take over. '
      'Notes §1.2 Monte Carlo combines SPC + SPAN to recover 100% of the\n'
      'proton core across the encounter.')

## Part 4: Thermal Protection — SPC under the 1400 °C Heat Shield / 1400 °C 차폐막 아래 SPC 열 보호

At closest approach (9.86 R$_\odot$, $S \approx 475 \times S_0$ where $S_0 = 1361$ W m$^{-2}$ is the AU value), the **PSP heat shield (TPS)** front face reaches $\sim$1400-1600 °C. SPC sits at the rim, partially exposed to the same flux. Notes §3.2 lists peak component temperatures: grid 1 > 1600 °C, modulator housing $\sim$1000 °C, collector plates $\sim$700 °C.

We solve the simplest energy-balance model: a thin component absorbing a fraction $\alpha$ of the incident solar flux and re-radiating with emissivity $\varepsilon$:

$$ \alpha\, S(r) \;=\; \varepsilon\, \sigma_\text{SB}\, T^4 + Q_\text{cond} \quad\Rightarrow\quad T = \left[\frac{\alpha S - Q_\text{cond}}{\varepsilon \sigma_\text{SB}}\right]^{1/4}, $$

where $S(r) = S_0 (1\,\text{AU}/r)^2$ and $Q_\text{cond}$ represents heat conducted away to the cold spacecraft body.

We sweep heliocentric distance from 0.7 AU (aphelion) down to 9.86 R$_\odot$ (perihelion) and check that:

1. The TPS front (high $\alpha$, low $\varepsilon$) reaches $\sim$1400-1600 °C at perihelion — close to the carbon-carbon limit.
2. The toy model under-counts conduction and view-factor effects, so it **over-estimates** SPC component temperatures vs the real flight values.
3. The qualitative ordering — TPS hottest, SPC grid 1 next, modulator housing third, collector plates coolest — matches notes §3.2.

복사 평형 식 $\alpha S = \varepsilon \sigma_\text{SB} T^4 + Q_\text{cond}$로 TPS 전면과 SPC 구성요소의 평형 온도를 9.86 R$_\odot$까지 거리 함수로 계산합니다. 단순 toy 모델은 실제보다 SPC 구성요소 온도를 과대평가하지만, 정성적 순서 (TPS > grid 1 > modulator > collector) 는 일치합니다.

In [ ]:
def equilibrium_temperature(r_AU: float,
                              alpha_solar: float,
                              epsilon_IR: float,
                              Q_cond_per_S0: float = 0.0) -> float:
    """Solve radiative-equilibrium temperature given material properties.

    Args:
        r_AU: Heliocentric distance [AU].
        alpha_solar: Solar-absorptance (0..1).
        epsilon_IR: IR emissivity (0..1).
        Q_cond_per_S0: Conducted heat loss expressed as a fraction of S_0 =
            1361 W m^-2 (positive removes heat).

    Returns:
        Equilibrium temperature in kelvin.
    """
    S0 = 1361.0
    S_local = S0 / r_AU ** 2
    flux_net = alpha_solar * S_local - Q_cond_per_S0 * S0
    flux_net = max(flux_net, 1.0e-3)
    return (flux_net / (epsilon_IR * SIGMA_SB)) ** 0.25


# Material properties (representative, not flight values).
components = {
    'TPS front (carbon-carbon)':      dict(alpha=0.95, eps=0.85, Q=0.00, color='C3'),
    'SPC W grid 1 (sunlit)':          dict(alpha=0.50, eps=0.30, Q=0.00, color='C0'),
    'SPC modulator housing (TZM)':    dict(alpha=0.40, eps=0.40, Q=0.05, color='C2'),
    'SPC collector plates (Nb)':      dict(alpha=0.30, eps=0.50, Q=0.10, color='C4'),
    'Spacecraft bus (radiator)':      dict(alpha=0.10, eps=0.85, Q=0.20, color='C8'),
}
material_limits_C = {
    'TPS front (carbon-carbon)':      1650.0,  # carbon-carbon limit
    'SPC W grid 1 (sunlit)':          3400.0,  # tungsten m.p. 3422 C
    'SPC modulator housing (TZM)':    1400.0,  # TZM working limit
    'SPC collector plates (Nb)':       900.0,  # niobium operating
    'Spacecraft bus (radiator)':       100.0,
}

r_AU_grid = np.linspace(0.7, 9.86 / 215.0, 200)  # 215 R_sun ≈ 1 AU

fig, ax = plt.subplots(figsize=(10, 6))
for name, props in components.items():
    T_K = np.array([equilibrium_temperature(r, props['alpha'], props['eps'],
                                              props['Q']) for r in r_AU_grid])
    T_C = T_K - 273.15
    ax.plot(r_AU_grid * 215.0, T_C, lw=2.0, color=props['color'], label=name)
    # Draw material limit as dotted line in same colour.
    ax.axhline(material_limits_C[name], ls=':', color=props['color'],
               alpha=0.4)

ax.axvline(9.86, ls='--', color='k', alpha=0.6, label='Perihelion 9.86 R$_\\odot$')
ax.axvspan(35.7, 9.86, color='C3', alpha=0.07)
ax.set_xlabel('Heliocentric distance [R$_\\odot$]')
ax.set_ylabel('Equilibrium temperature [°C]')
ax.set_title('SPC and TPS equilibrium temperature vs distance\n거리별 SPC·TPS 평형 온도')
ax.set_xscale('log')
ax.set_yscale('symlog', linthresh=10.0)
ax.set_xlim(150.0, 9.0)
ax.invert_xaxis()
ax.grid(alpha=0.3, which='both')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

# Tabulate the perihelion temperatures.
r_peri = 9.86 / 215.0
print(f'Perihelion distance: 9.86 R_sun = {r_peri:.4f} AU')
print(f'Local solar flux   : {1361.0 / r_peri ** 2:.0f} W/m^2 = '
      f'{1361.0 / r_peri ** 2 / 1361.0:.0f} × Earth value')
print(f'\n{"Component":40s} {"T_eq [°C]":>10s} {"Limit [°C]":>12s} '
      f'{"Margin [°C]":>13s}')
print('-' * 78)
n_over_limit = 0
for name, props in components.items():
    T_K = equilibrium_temperature(r_peri, props['alpha'], props['eps'],
                                    props['Q'])
    T_C = T_K - 273.15
    limit = material_limits_C[name]
    margin = limit - T_C
    if margin < 0:
        n_over_limit += 1
    print(f'{name:40s} {T_C:10.0f} {limit:12.0f} {margin:13.0f}')
print(f'\n{n_over_limit} of {len(components)} components exceed their limits in this'
       ' simple radiative-equilibrium toy model.')
print('Real SPP/SWEAP design adds: (1) active conduction to the cold spacecraft\n'
      '  bus through TPS struts, (2) selective optical filtering by the front\n'
      '  niobium shield plates that limit FSU sunlight exposure, (3) sapphire\n'
      '  insulators that retain high resistivity at high T, (4) etched single-\n'
      '  crystal tungsten grids replacing wire mesh. These engineering choices\n'
      '  are what reduce the actual component temperatures (notes §3.2: grid 1\n'
      '  >1600 °C, modulator housing ~1000 °C, plates ~700 °C) below the\n'
      '  values predicted by this toy.')
print('실제 SPP/SWEAP는 능동 전도, niobium shield plate, sapphire 절연체, single-\n'
      '  crystal tungsten 격자로 위 toy 모델의 평형 온도를 낮춰 실제 9.86 R_sun에서\n'
      '  modulator housing ~1000°C, plates ~700°C 까지 유지합니다.')

### 4.b Why AC modulation matters at 1400 °C / 1400 °C에서 AC 변조의 의미

Hot metal surfaces emit electrons via **thermionic (Richardson-Dushman) emission**:

$$ J_\text{th} \;=\; A_R\, T^2\, \exp\!\left(-\frac{\phi}{k_B T}\right), \qquad A_R \approx 1.20 \times 10^{6}\ \text{A m}^{-2}\,\text{K}^{-2}, $$

where $\phi$ is the work function. For the SPC modulator housing at $\sim$1000 °C, even a square-millimetre of exposed metal can emit a current orders of magnitude above the proton plate current — a fatal DC noise source if SPC used DC measurement.

However, thermionic emission is essentially **DC**: it does not vary at the modulator frequency $f_\text{mod} = 1280$ Hz. SPC's lock-in detection at $f_\text{mod}$ rejects this DC component along with photoelectrons and SEP ionisation currents. We compute the thermionic current density at component temperatures and confirm it dwarfs the signal current — yet is invisible to the AC detector.

고온 금속 표면은 Richardson-Dushman 방정식에 따라 thermionic 전자를 방출합니다. SPC modulator housing ($\sim$1000 °C)에서 이 DC 전류는 양성자 신호 전류보다 수 차수 큽니다. 그러나 thermionic 방출은 본질적으로 DC이므로 1280 Hz lock-in 검파가 자연스럽게 거릅니다. 동일한 원리로 SEP 관통 잡음과 광전자 잡음도 제거됩니다.

In [ ]:
def thermionic_current_density(T_K: np.ndarray,
                                  work_function_eV: float = 4.5) -> np.ndarray:
    """Compute the Richardson-Dushman thermionic current density.

    Args:
        T_K: Temperature [K].
        work_function_eV: Work function phi [eV]. Default 4.5 (≈ tungsten).

    Returns:
        Current density J_th in A m^-2.
    """
    A_R = 1.20173e6
    phi_J = work_function_eV * EV_TO_J
    return A_R * T_K ** 2 * np.exp(-phi_J / (K_B * T_K))


T_C_grid = np.linspace(300.0, 1800.0, 200)
T_K_grid = T_C_grid + 273.15
J_th = thermionic_current_density(T_K_grid, work_function_eV=4.5)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(T_C_grid, J_th, 'C3-', lw=2.0)
# Mark component temperatures from Part 4.
for name, T_C in zip(["Collector plates ~700 °C",
                       "Modulator housing ~1000 °C",
                       "Grid 1 ~1600 °C"],
                      [700.0, 1000.0, 1600.0]):
    J_at = thermionic_current_density(np.array([T_C + 273.15]),
                                       work_function_eV=4.5)[0]
    ax.plot(T_C, J_at, 'o', ms=10)
    ax.annotate(name, xy=(T_C, J_at),
                xytext=(T_C - 100, J_at * 30),
                fontsize=9,
                arrowprops=dict(arrowstyle='-', color='gray'))
ax.set_xlabel('Surface temperature [°C]')
ax.set_ylabel('Thermionic current density J_th [A m⁻²]')
ax.set_title('Richardson-Dushman thermionic emission (φ = 4.5 eV)\nRichardson-Dushman thermionic 방출')
ax.set_yscale('log')
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

# Compare to typical SPC signal current at 9.86 R_sun.
A_collector_m2 = 5.0e-4    # 5 cm^2
A_grid_emit_m2 = 1.0e-6    # 1 mm^2 of emitting metal
I_signal = I_V.max()        # from Part 1
I_th_collector = thermionic_current_density(700.0 + 273.15)[()] * A_collector_m2 \
    if False else thermionic_current_density(np.array([700.0 + 273.15]))[0] \
    * A_collector_m2
I_th_modulator = thermionic_current_density(np.array([1000.0 + 273.15]))[0] \
    * A_grid_emit_m2
print(f'SPC signal current (9.86 R_sun)        : {I_signal:.2e} A')
print(f'Thermionic from collector at 700 °C    : {I_th_collector:.2e} A')
print(f'Thermionic from 1 mm² grid at 1000 °C  : {I_th_modulator:.2e} A')
print('→ DC thermionic noise dwarfs the signal but is rejected by 1280 Hz lock-in.')

## Summary / 요약

SWEAP combines a sun-staring Faraday cup (SPC) with two top-hat ESAs (SPAN-A on the ram side, SPAN-B anti-ram) to recover the full proton and electron VDF inside 0.25 AU despite the 1400 °C heat-shield FOV blockage. This notebook shows in toy form the four design pillars: the SPC I-V curve and AC synchronous detection that lets it operate at 9.86 R$_\odot$; the top-hat ESA's 7% energy and ±60° elevation response; the kinetic moment integration that reduces a 3-D non-Maxwellian VDF to standard plasma parameters; and the thermal balance that keeps SPC components within their material limits.

SWEAP는 차폐막 가장자리의 태양 직시 Faraday Cup (SPC)과 두 ESA (SPAN-A 램, SPAN-B 반-램)를 결합해 0.25 AU 안쪽에서 양성자·전자 VDF를 완전 복원합니다. 본 노트북은 네 가지 설계 핵심 — 9.86 R$_\odot$에서의 SPC I-V 곡선, top-hat ESA 응답, 비-맥스웰 VDF 모멘트 적분, 열 보호 — 을 toy 모델로 재현했습니다.

| Concept / 개념 | SWEAP / Kasper 2016 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Sun-staring Faraday cup / 태양 직시 FC | SPC, 1280 Hz mod, ±28° FOV, 5×10⁻¹³ – 10⁻⁷ A | DSCOVR Faraday Cup; future ESCAPADE FCs |
| Top-hat hemispherical ESA / Top-hat ESA | SPAN-A/B, ΔR/R = 0.03, 240°×120° | Solar Orbiter SWA-PAS/EAS; THEMIS ESA; MAVEN STATIC |
| TOF mass spectrometer / TOF 질량 분광 | SPAN-A ion section, 15 kV pre-accel | MMS HPCA; MAVEN STATIC |
| AC synchronous detection / AC 동기 검파 | 1280 Hz lock-in for SPC | Wind/SWE FC heritage; ULF/ELF E-field receivers |
| L3 moment pipeline / L3 모멘트 | Kasper 2006 χ² fit (RDF + 4-collector) | SPEDAS, PySPEDAS PSP modules |
| Thermal protection / 열 보호 | TPS + sapphire/Nb/W/TZM materials | Solar Orbiter heat shield; ESA Smile |
| Aberration handling / aberration 처리 | SPC ±28° + SPAN-A ram FOV | Bepi/Mio MIA; future Helioswarm |
| Predecessor / 선구 | Helios FC (1974); Wind/SWE FC (1995); FAST/THEMIS ESA (1996/2007) | — |
| Successor / 후계 | Solar Orbiter SWA (2020); ESCAPADE (2024); Interstellar Probe (TBD) | — |

**Key Takeaway / 핵심 시사점**: The SPC + SPAN combination is not an incremental upgrade but a **systems-level answer to the heat-shield FOV constraint**. The Faraday cup gives proton/alpha bulk-flow physics even when mounted at 700 °C in a 5×10⁵ SEP-flux environment, while the top-hat ESAs give 3-D VDFs over the 4π sky behind the shield. Together they recover proton core 100% and electron strahl 98% of the time at the final 9.86 R$_\odot$ orbit — the physics behind every PSP switchback, sub-Alfvénic crossing, and ion heating result.

**핵심 시사점**: SPC + SPAN 결합은 점진적 업그레이드가 아니라 차폐막 FOV 제약에 대한 시스템 차원의 해법입니다. Faraday cup은 700 °C에 노출된 5×10⁵ SEP 환경에서도 양성자·알파 흐름 물리를 제공하고, top-hat ESA는 차폐막 뒤 4π 하늘에서 3-D VDF를 제공합니다. 두 측정기의 결합으로 9.86 R$_\odot$ 궤도에서 양성자 core 100%, 전자 strahl 98%를 확보 — PSP의 switchback, sub-Alfvénic crossing, ion heating 발견의 기반.